# Reset Notebook Configurations

In [ ]:
# Restart the Runtime (Hard Reset)
# This is the most effective way to completely clear RAM and disk cache.
import os
os.kill(os.getpid(), 9)

In [ ]:
# Delete variables
%reset -f

# Clear CUDA cache (only for deep learning examples)
import torch
torch.cuda.empty_cache()

# Clear garbage
import gc
gc.collect()

30

In [ ]:
!rm -rf /content/*
!rm -rf ~/.cache/huggingface

In [ ]:
!df -h       # Disk usage
print("="*100)
print("="*100)
!nvidia-smi  # GPU usage
print("="*100)
print("="*100)
!free -h     # RAM usage

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   65G  43% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.2G  750M  62% /usr/sbin/docker-init
/dev/sda1       119G   72G   48G  61% /opt/bin/.nvidia
tmpfs           6.4G  7.6M  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
Fri Nov 14 04:26:33 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|              

# Code Run

In [5]:
# # Install uv package manager
!pip install uv

In [7]:
# ============================================================================
# STEP 1: Install Dependencies
# ============================================================================
print("Installing qwen-tts package...")
!uv pip install -q qwen-tts soundfile

# Optional: Install Flash Attention 2 for better performance (requires compatible GPU)
# Note: This may take a few minutes
print("Installing flash-attention (optional, for better performance)...")
!uv pip install -q flash-attn --no-build-isolation

Installing qwen-tts package...
Installing flash-attention (optional, for better performance)...


In [8]:
# ============================================================================
# STEP 2: Import Libraries
# ============================================================================
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel
from IPython.display import Audio, display

print("Libraries imported successfully!")

Libraries imported successfully!


In [9]:
# ============================================================================
# STEP 3: Load the Model
# ============================================================================
print("Loading Qwen3-TTS model...")
print("This may take a few minutes on first run as it downloads the model weights...")

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    device_map="cuda:0" if torch.cuda.is_available() else "cpu",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2" if torch.cuda.is_available() else "eager",
)

print("Model loaded successfully!")
print(f"Using device: {'CUDA (GPU)' if torch.cuda.is_available() else 'CPU'}")

Loading Qwen3-TTS model...
This may take a few minutes on first run as it downloads the model weights...


model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

configuration.json:   0%|          | 0.00/76.0 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Model loaded successfully!
Using device: CUDA (GPU)


In [15]:
# ============================================================================
# EXAMPLE 1: English Text-to-Speech
# ============================================================================
print("\n" + "="*80)
print("EXAMPLE 1: English Text-to-Speech")
print("="*80)

# Check supported speakers
print("\nSupported speakers:", model.get_supported_speakers())
print("Supported languages:", model.get_supported_languages())

# Generate English speech
english_text = "Hello! This is a demonstration of Qwen3 text-to-speech synthesis. The model can generate natural-sounding speech in multiple languages."
# english_text = "Artificial intelligence is a transformative field of technology because its impact is felt after systems are deployed and integrated into everyday life, rather than only at the moment of innovation. It focuses on creating machines and models that can learn, reason, and make decisions, often inspired by human intelligence but operating at vastly different scales. From data analysis and automation to creative generation and scientific discovery, artificial intelligence continues to reshape how industries function and how humans interact with technology, gradually redefining what tasks can be augmented or performed by machines."

print(f"\nGenerating speech for: '{english_text}'")

wavs_en, sr_en = model.generate_custom_voice(
    text=english_text,
    language="English",
    speaker="Ryan",  # Dynamic male voice with strong rhythmic drive
    instruct="Speak with enthusiasm and clarity"  # Optional instruction
)

# Save the audio file
output_file_en = "output_english.wav"
sf.write(output_file_en, wavs_en[0], sr_en)
print(f"English audio saved to: {output_file_en}")

# Play the audio in Colab
print("\nPlaying English audio:")
display(Audio(wavs_en[0], rate=sr_en))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



EXAMPLE 1: English Text-to-Speech

Supported speakers: ['aiden', 'dylan', 'eric', 'ono_anna', 'ryan', 'serena', 'sohee', 'uncle_fu', 'vivian']
Supported languages: ['auto', 'chinese', 'english', 'french', 'german', 'italian', 'japanese', 'korean', 'portuguese', 'russian', 'spanish']

Generating speech for: 'Artificial intelligence is a transformative field of technology because its impact is felt after systems are deployed and integrated into everyday life, rather than only at the moment of innovation. It focuses on creating machines and models that can learn, reason, and make decisions, often inspired by human intelligence but operating at vastly different scales. From data analysis and automation to creative generation and scientific discovery, artificial intelligence continues to reshape how industries function and how humans interact with technology, gradually redefining what tasks can be augmented or performed by machines.'
English audio saved to: output_english.wav

Playing Engl

In [12]:
# ============================================================================
# EXAMPLE 2: Japanese Text-to-Speech
# ============================================================================
print("\n" + "="*80)
print("EXAMPLE 2: Japanese Text-to-Speech")
print("="*80)

# Generate Japanese speech
japanese_text = "こんにちは！これはQwen3音声合成のデモンストレーションです。このモデルは多言語で自然な音声を生成できます。"
# japanese_text = "人工知能は、革新的な瞬間そのものよりも、システムが実際に導入され日常生活に組み込まれた後にその影響が現れるという点で、変革的な技術分野です。人工知能は、人間の知性に着想を得ながらもまったく異なる規模で動作し、学習し、推論し、意思決定を行うことができる機械やモデルの構築に焦点を当てています。データ分析や自動化から創造的な生成、科学的発見に至るまで、人工知能は産業のあり方や人間と技術との関わり方を変え続け、機械が補助または実行できるタスクの概念を徐々に再定義しています。"

print(f"\nGenerating speech for: '{japanese_text}'")

wavs_ja, sr_ja = model.generate_custom_voice(
    text=japanese_text,
    language="Japanese",
    speaker="Ono_Anna",  # Playful Japanese female voice with light, nimble timbre
    instruct="話し方は明るく元気に"  # Optional: Speak brightly and energetically
)

# Save the audio file
output_file_ja = "output_japanese.wav"
sf.write(output_file_ja, wavs_ja[0], sr_ja)
print(f"Japanese audio saved to: {output_file_ja}")

# Play the audio in Colab
print("\nPlaying Japanese audio:")
display(Audio(wavs_ja[0], rate=sr_ja))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



EXAMPLE 2: Japanese Text-to-Speech

Generating speech for: '人工知能は、革新的な瞬間そのものよりも、システムが実際に導入され日常生活に組み込まれた後にその影響が現れるという点で、変革的な技術分野です。人工知能は、人間の知性に着想を得ながらもまったく異なる規模で動作し、学習し、推論し、意思決定を行うことができる機械やモデルの構築に焦点を当てています。データ分析や自動化から創造的な生成、科学的発見に至るまで、人工知能は産業のあり方や人間と技術との関わり方を変え続け、機械が補助または実行できるタスクの概念を徐々に再定義しています。'
Japanese audio saved to: output_japanese.wav

Playing Japanese audio:


In [14]:
# ============================================================================
# EXAMPLE 3: Batch Generation (Multiple Sentences)
# ============================================================================
print("\n" + "="*80)
print("EXAMPLE 3: Batch Generation (English + Japanese)")
print("="*80)

# Generate multiple sentences at once
texts = [
    "The weather today is absolutely wonderful!",
    "今日の天気は本当に素晴らしいですね！"
]

languages = ["English", "Japanese"]
speakers = ["Aiden", "Ono_Anna"]  # Sunny American male, Playful Japanese female
instructions = ["Very happy and excited", "嬉しそうに、わくわくした感じで"]

print("\nGenerating batch speech...")
wavs_batch, sr_batch = model.generate_custom_voice(
    text=texts,
    language=languages,
    speaker=speakers,
    instruct=instructions
)

# Save and play each audio
for i, (wav, txt, lang) in enumerate(zip(wavs_batch, texts, languages)):
    output_file = f"output_batch_{i}_{lang.lower()}.wav"
    sf.write(output_file, wav, sr_batch)
    print(f"\n[{i+1}] {lang}: '{txt}'")
    print(f"    Saved to: {output_file}")
    display(Audio(wav, rate=sr_batch))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



EXAMPLE 3: Batch Generation (English + Japanese)

Generating batch speech...

[1] English: 'The weather today is absolutely wonderful!'
    Saved to: output_batch_0_english.wav



[2] Japanese: '今日の天気は本当に素晴らしいですね！'
    Saved to: output_batch_1_japanese.wav


In [13]:
# ============================================================================
# SPEAKER INFORMATION
# ============================================================================
print("\n" + "="*80)
print("AVAILABLE SPEAKERS")
print("="*80)
print("""
English Speakers:
- Ryan: Dynamic male voice with strong rhythmic drive
- Aiden: Sunny American male voice with a clear midrange

Japanese Speakers:
- Ono_Anna: Playful Japanese female voice with a light, nimble timbre

Chinese Speakers:
- Vivian: Bright, slightly edgy young female voice
- Serena: Warm, gentle young female voice
- Uncle_Fu: Seasoned male voice with a low, mellow timbre
- Dylan: Youthful Beijing male voice (Beijing Dialect)
- Eric: Lively Chengdu male voice (Sichuan Dialect)

Korean Speakers:
- Sohee: Warm Korean female voice with rich emotion

Note: Each speaker can speak any language supported by the model,
but they perform best in their native language.
""")

print("\n" + "="*80)
print("Quickstart complete! You can now modify the text and parameters above.")
print("="*80)


AVAILABLE SPEAKERS

English Speakers:
- Ryan: Dynamic male voice with strong rhythmic drive
- Aiden: Sunny American male voice with a clear midrange

Japanese Speakers:
- Ono_Anna: Playful Japanese female voice with a light, nimble timbre

Chinese Speakers:
- Vivian: Bright, slightly edgy young female voice
- Serena: Warm, gentle young female voice
- Uncle_Fu: Seasoned male voice with a low, mellow timbre
- Dylan: Youthful Beijing male voice (Beijing Dialect)
- Eric: Lively Chengdu male voice (Sichuan Dialect)

Korean Speakers:
- Sohee: Warm Korean female voice with rich emotion

Note: Each speaker can speak any language supported by the model,
but they perform best in their native language.


Quickstart complete! You can now modify the text and parameters above.
